
# Test created tokenizers

In [1]:
from multiprocess import Process, Queue
import os
import time

In [2]:
def parallel_count(input_queue:Queue, 
                   output_queue:Queue,
                   counter_id:str):

    from datasets import load_dataset
    from transformers import PreTrainedTokenizerFast

    def count_tokens(tokenizer, dataset, text_column='text', max_samples=None):
        """Each worker iterates the ENTIRE dataset but only counts tokens for its tokenizer_subset."""
        local_counts = 0
        sample_count = 0
    
        if not max_samples: max_samples = dataset.num_rows
        max_samples = min(max_samples, dataset.num_rows)

    
        for row in dataset.iter(1):
            text = row[text_column][0]
            if not isinstance(text, str):
                continue
            local_counts += len(tokenizer.encode(text, add_special_tokens=False))
            sample_count += 1
            if sample_count == max_samples:
                break
    
        return local_counts, sample_count
    
    
    def load_and_count(tokenizer_path:str, 
                   dataset_name:str='TucanoBR/wikipedia-PT', 
                   dataset_split:str='train',
                   cache_dir:str='../data',
                   text_column='text', 
                   max_samples=None) -> dict:
    
        tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_path)
        dataset = load_dataset(dataset_name, split=dataset_split, cache_dir=cache_dir)
    
        token_count, sample_size = count_tokens(tokenizer, dataset)
    
        result = {'name': tokenizer_path,
                  'token_count': token_count,
                  'sample_size': sample_size}
        
        return result
    
    while True:
        item = input_queue.get()
        if item is None:  # Sentinel value to exit
            print(f"Counter {counter_id} exiting")
            output_queue.put(None)
            break
        print(f"Counter {counter_id} processing item: {item}")
        result = load_and_count(item)
        result['counter_id'] = counter_id
        output_queue.put(result)
        print(result)

In [3]:
start_time = time.time()

tokenizer_paths = [file for file in os.listdir() if file.endswith('.json')]
num_consumers = 5
input_queue = Queue()
output_queue = Queue()

for tokenizer_path in tokenizer_paths:
    print(f'add tokenizer: {tokenizer_path} to queue')
    input_queue.put(tokenizer_path)

consumers = []
for i in range(num_consumers):
    print(f'Starting token counter {i}')
    p = Process(target=parallel_count, args=(input_queue, output_queue, i))
    consumers.append(p)
    p.start()

# Add sentinel values (one per consumer) to signal exit
for _ in range(num_consumers):
    input_queue.put(None)

# Collect results from the results queue
results = []
sentinel_count = 0
while sentinel_count < num_consumers:
    result = output_queue.get()  # Blocks until a result is available
    if result is None:  # Check for sentinel
        sentinel_count += 1
    else:
        print(f"Received result: {result}")
        results.append(result)

for p in consumers:
        p.join()

total_time = time.time() - start_time
print(f'Total time: {total_time}')

add tokenizer: portuguese_bpe_tokenizer_1.json to queue
add tokenizer: portuguese_bpe_tokenizer_2.json to queue
add tokenizer: portuguese_bpe_tokenizer_3.json to queue
add tokenizer: portuguese_bpe_tokenizer_4.json to queue
add tokenizer: portuguese_bpe_tokenizer_5.json to queue
add tokenizer: portuguese_bpe_tokenizer_6.json to queue
add tokenizer: portuguese_bpe_tokenizer_7.json to queue
add tokenizer: portuguese_bpe_tokenizer_8.json to queue
add tokenizer: portuguese_bpe_tokenizer_9.json to queue
Starting counter 0
Starting counter 1
Starting counter 2
Starting counter 3
Starting counter 4
Received result: {'name': 'portuguese_bpe_tokenizer_4.json', 'token_count': 674601344, 'sample_size': 1103446, 'counter_id': 3}
Received result: {'name': 'portuguese_bpe_tokenizer_5.json', 'token_count': 661595424, 'sample_size': 1103446, 'counter_id': 4}
Received result: {'name': 'portuguese_bpe_tokenizer_1.json', 'token_count': 608321068, 'sample_size': 1103446, 'counter_id': 0}
Received result: 

In [4]:
for result in results:
    print(result)

{'name': 'portuguese_bpe_tokenizer_4.json', 'token_count': 674601344, 'sample_size': 1103446, 'counter_id': 3}
{'name': 'portuguese_bpe_tokenizer_5.json', 'token_count': 661595424, 'sample_size': 1103446, 'counter_id': 4}
{'name': 'portuguese_bpe_tokenizer_1.json', 'token_count': 608321068, 'sample_size': 1103446, 'counter_id': 0}
{'name': 'portuguese_bpe_tokenizer_3.json', 'token_count': 603705401, 'sample_size': 1103446, 'counter_id': 2}
{'name': 'portuguese_bpe_tokenizer_2.json', 'token_count': 604103063, 'sample_size': 1103446, 'counter_id': 1}
{'name': 'portuguese_bpe_tokenizer_6.json', 'token_count': 661736917, 'sample_size': 1103446, 'counter_id': 3}
{'name': 'portuguese_bpe_tokenizer_9.json', 'token_count': 560224561, 'sample_size': 1103446, 'counter_id': 2}
{'name': 'portuguese_bpe_tokenizer_8.json', 'token_count': 559658771, 'sample_size': 1103446, 'counter_id': 0}
{'name': 'portuguese_bpe_tokenizer_7.json', 'token_count': 561148010, 'sample_size': 1103446, 'counter_id': 4}


In [5]:
start_time = time.time()

tokenizer_paths = [file for file in os.listdir() if file.endswith('.json')]
num_consumers = 9
input_queue = Queue()
output_queue = Queue()

for tokenizer_path in tokenizer_paths:
    print(f'add tokenizer: {tokenizer_path} to queue')
    input_queue.put(tokenizer_path)

consumers = []
for i in range(num_consumers):
    print(f'Starting token counter {i}')
    p = Process(target=parallel_count, args=(input_queue, output_queue, i))
    consumers.append(p)
    p.start()

# Add sentinel values (one per consumer) to signal exit
for _ in range(num_consumers):
    input_queue.put(None)

# Collect results from the results queue
results = []
sentinel_count = 0
while sentinel_count < num_consumers:
    result = output_queue.get()  # Blocks until a result is available
    if result is None:  # Check for sentinel
        sentinel_count += 1
    else:
        print(f"Received result: {result}")
        results.append(result)

for p in consumers:
        p.join()

total_time = time.time() - start_time
print(f'Total time: {total_time}')

add tokenizer: portuguese_bpe_tokenizer_1.json to queue
add tokenizer: portuguese_bpe_tokenizer_2.json to queue
add tokenizer: portuguese_bpe_tokenizer_3.json to queue
add tokenizer: portuguese_bpe_tokenizer_4.json to queue
add tokenizer: portuguese_bpe_tokenizer_5.json to queue
add tokenizer: portuguese_bpe_tokenizer_6.json to queue
add tokenizer: portuguese_bpe_tokenizer_7.json to queue
add tokenizer: portuguese_bpe_tokenizer_8.json to queue
add tokenizer: portuguese_bpe_tokenizer_9.json to queue
Starting token counter 0
Starting token counter 1
Starting token counter 2
Starting token counter 3
Starting token counter 4
Starting token counter 5
Starting token counter 6
Starting token counter 7
Starting token counter 8
Received result: {'name': 'portuguese_bpe_tokenizer_4.json', 'token_count': 674601344, 'sample_size': 1103446, 'counter_id': 3}
Received result: {'name': 'portuguese_bpe_tokenizer_5.json', 'token_count': 661595424, 'sample_size': 1103446, 'counter_id': 4}
Received result

| Name | Vocab Size | sample size |   token count  |
|------|------------|-------------|----------------|
| 1    |   30,000   | 679,609     | 608,321,068    |
| 2    |   30,000   | 657,685     | 604,103,063    |
| 3    |   30,000   | 200,000     | 603,705,401    |
| 4    |   15,000   | 679,609     | 674,601,344    |
| 5    |   15,000   | 657,685     | 661,595,424    |
| 6    |   15,000   | 200,000     | 661,736,917    |
| 7    |   60,000   | 679,609     | 561,148,010    |
| 8    |   60,000   | 657,685     | 559,658,771    |
| 9    |   60,000   | 200,000     | 560,224,561    |